# Reinforcement Learning with Verifiable Rewards (RLVR)
## GRPO & Test-Time Scaling: Thinking vs Non-Thinking Models

In this exercise session, you will:

1. **Understand the mathematical foundations** of Group Relative Policy Optimization (GRPO)
2. **Implement GRPO from scratch** — step by step, component by component
3. **Build a verifiable reward function** for mathematical reasoning tasks
4. **Explore test-time compute scaling** by comparing thinking vs non-thinking modes of Qwen3-4B
5. **Evaluate on AIME 2024 & 2025** and visualize how thinking budget affects performance

**References:**
- [DeepSeekMath (Shao et al., 2024)](https://arxiv.org/abs/2402.03300) — introduced GRPO
- [DeepSeek-R1 (DeepSeek, 2025)](https://arxiv.org/abs/2501.12948) — RLVR at scale
- [Qwen3 Technical Report](https://arxiv.org/abs/2505.09388) — thinking / non-thinking modes
- [Scaling LLM Test-Time Compute (Snell et al., 2024)](https://arxiv.org/abs/2408.03314)


---
## Part 0: Setup

Install the required packages and import dependencies.

In [ ]:
# Run this cell to install dependencies (uncomment if needed)
# ! pip install torch transformers datasets accelerate matplotlib numpy tqdm


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Optional
import re
import warnings
warnings.filterwarnings('ignore')

# Verify GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


---
## Part 1: Mathematical Foundations of GRPO

### 1.1 From RLHF to RLVR

**Reinforcement Learning from Human Feedback (RLHF)** trains a reward model from human preferences,
then uses RL (typically PPO) to optimize a language model against that learned reward. This has two drawbacks:

1. **Reward model noise** — learned rewards are imperfect proxies for human preferences
2. **Reward hacking** — the policy can exploit reward model weaknesses

**Reinforcement Learning with Verifiable Rewards (RLVR)** sidesteps both issues for domains where
correctness can be verified programmatically:

$$r(q, o) = \begin{cases} 1 & \text{if } \texttt{extract\_answer}(o) = \text{ground\_truth}(q) \\ 0 & \text{otherwise} \end{cases}$$

Mathematics is a natural fit — AIME answers are integers in $[0, 999]$, so verification is exact.


### 1.2 From PPO to GRPO: Removing the Critic

**PPO** (Proximal Policy Optimization) is the standard RL algorithm used in RLHF.
It requires a **value function** (critic) $V_\phi(s)$ to estimate advantages:

$$\hat{A}_t^{\text{GAE}} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}, \quad \delta_t = r_t + \gamma V_\phi(s_{t+1}) - V_\phi(s_t)$$

Training a value network that is as large as the policy **doubles** the memory and compute cost.

**Key Insight of GRPO:** Instead of learning a value function, sample a **group** of $G$ outputs
for each prompt and use the group statistics as a baseline:

$$\hat{A}_i = \frac{r_i - \text{mean}(\{r_j\}_{j=1}^G)}{\text{std}(\{r_j\}_{j=1}^G) + \varepsilon}$$

This simple normalization:
- Eliminates the value network entirely (~50% memory savings)
- Uses the **outcome-level** reward (the full response is correct or not)
- All tokens in the same response share the same advantage $\hat{A}_i$


### 1.3 The GRPO Objective

For a batch of prompts $q$, GRPO samples $G$ outputs $\{o_1, \ldots, o_G\}$ from
the old policy $\pi_{\theta_{\text{old}}}(\cdot|q)$ and optimizes:

$$\mathcal{J}_{\text{GRPO}}(\theta) = \mathbb{E}_{q \sim \mathcal{D}} \left[ \frac{1}{G} \sum_{i=1}^{G} \frac{1}{|o_i|} \sum_{t=1}^{|o_i|} \Big( \min\big( \rho_{i,t}(\theta)\, \hat{A}_i,\; \text{clip}(\rho_{i,t}(\theta), 1{-}\epsilon, 1{+}\epsilon)\, \hat{A}_i \big) - \beta\, D_{\text{KL}}^{(i,t)} \Big) \right]$$

where each component is defined as:

| Symbol | Definition |
|--------|-----------|
| $\rho_{i,t}(\theta)$ | Policy ratio: $\dfrac{\pi_\theta(o_{i,t} \mid q, o_{i,<t})}{\pi_{\theta_{\text{old}}}(o_{i,t} \mid q, o_{i,<t})}$ |
| $\hat{A}_i$ | Group-normalized advantage (shared across all tokens in output $o_i$) |
| $\epsilon$ | Clipping parameter (typically 0.2) |
| $\beta$ | KL penalty coefficient |
| $D_{\text{KL}}^{(i,t)}$ | Per-token KL divergence estimate between $\pi_\theta$ and $\pi_{\text{ref}}$ |

The per-token **KL divergence** uses the following unbiased estimator:

$$D_{\text{KL}}^{(i,t)} = \frac{\pi_{\text{ref}}(o_{i,t} \mid q, o_{i,<t})}{\pi_\theta(o_{i,t} \mid q, o_{i,<t})} - \log \frac{\pi_{\text{ref}}(o_{i,t} \mid q, o_{i,<t})}{\pi_\theta(o_{i,t} \mid q, o_{i,<t})} - 1$$

In log-probability space, letting $\delta_t = \log \pi_{\text{ref}} - \log \pi_\theta$:

$$D_{\text{KL}}^{(i,t)} = e^{\delta_t} - \delta_t - 1$$


### 1.4 GRPO Algorithm Overview

```
Algorithm: GRPO (Group Relative Policy Optimization)
────────────────────────────────────────────────────
Input: Policy π_θ, reference policy π_ref, dataset D
       Group size G, clip ε, KL weight β, epochs M

For each iteration:
  1. Sample batch of prompts {q_1, ..., q_B} from D
  2. For each prompt q_i:
     a. Sample G outputs: {o_1, ..., o_G} ~ π_θ_old(·|q_i)
     b. Compute rewards: {r_1, ..., r_G} via verifiable reward
     c. Normalize advantages: A_j = (r_j - mean(r)) / (std(r) + ε)
     d. Compute log-probs under π_θ_old and π_ref
  3. For M optimization epochs:
     a. Compute log-probs under current π_θ
     b. Compute token-level ratios: ρ_t = exp(log π_θ - log π_θ_old)
     c. Compute clipped surrogate objective
     d. Compute KL penalty against π_ref
     e. Update θ by gradient ascent on J_GRPO
  4. Set π_θ_old ← π_θ
```

---
## Part 2: Implementing GRPO — Step by Step

We will now implement each component of GRPO. For each exercise:
- Read the description and formula carefully
- Fill in the code where you see `### YOUR CODE HERE ###`
- Run the test cell to verify your implementation


### Exercise 1: Computing Per-Token Log Probabilities

Given model logits and target token IDs, compute the log probability of each token.

$$\log \pi_\theta(o_t \mid q, o_{<t}) = \log \text{softmax}(\text{logits}_t)[o_t] = \text{logits}_t[o_t] - \log \sum_v \exp(\text{logits}_t[v])$$

**Input:**
- `logits`: Tensor of shape `(batch_size, seq_len, vocab_size)`
- `labels`: Tensor of shape `(batch_size, seq_len)` — the target token IDs

**Output:**
- `log_probs`: Tensor of shape `(batch_size, seq_len)` — per-token log probabilities

**Hint:** Use `torch.log_softmax` and `torch.gather`.

In [ ]:
def compute_log_probs(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """
    Compute per-token log probabilities from logits.

    Args:
        logits: (batch_size, seq_len, vocab_size)
        labels: (batch_size, seq_len)
    Returns:
        log_probs: (batch_size, seq_len)
    """
    ### SOLUTION ###
    log_probs = F.log_softmax(logits, dim=-1)
    log_probs = torch.gather(log_probs, 2, labels.unsqueeze(-1)).squeeze(-1)
    return log_probs
    ### END SOLUTION ###


**Test your implementation:**

In [ ]:
# ── Test: compute_log_probs ──
torch.manual_seed(42)
_logits = torch.randn(2, 3, 5)  # batch=2, seq=3, vocab=5
_labels = torch.tensor([[0, 2, 4], [1, 3, 0]])
_log_probs = compute_log_probs(_logits, _labels)

# Manual computation for verification
_log_softmax = torch.log_softmax(_logits, dim=-1)
_expected = torch.gather(_log_softmax, 2, _labels.unsqueeze(-1)).squeeze(-1)

assert _log_probs.shape == (2, 3), f'Expected shape (2,3), got {_log_probs.shape}'
assert torch.allclose(_log_probs, _expected, atol=1e-6), 'Values do not match expected'
assert (_log_probs <= 0).all(), 'Log probabilities must be <= 0'
print('✓ Exercise 1 passed!')


### Exercise 2: Group-Normalized Advantage Estimation

Given rewards for a group of $G$ outputs sampled for the **same prompt**, compute
the normalized advantages:

$$\hat{A}_i = \frac{r_i - \mu}{\sigma + \varepsilon}, \quad \mu = \frac{1}{G}\sum_{j=1}^G r_j, \quad \sigma = \text{std}(\{r_j\}_{j=1}^G)$$

**Edge case:** If all rewards are identical ($\sigma = 0$), all advantages should be 0.

**Input:**
- `rewards`: Tensor of shape `(batch_size, group_size)` — reward per output
- `eps`: Small constant for numerical stability (default: 1e-8)

**Output:**
- `advantages`: Tensor of shape `(batch_size, group_size)`

In [ ]:
def compute_group_advantages(
    rewards: torch.Tensor,
    eps: float = 1e-8,
) -> torch.Tensor:
    """
    Compute group-normalized advantages from rewards.

    Args:
        rewards: (batch_size, group_size)
        eps: numerical stability constant
    Returns:
        advantages: (batch_size, group_size)
    """
    ### SOLUTION ###
    mean = rewards.mean(dim=-1, keepdim=True)
    std = rewards.std(dim=-1, keepdim=True)
    return (rewards - mean) / (std + eps)
    ### END SOLUTION ###


**Test your implementation:**

In [ ]:
# ── Test: compute_group_advantages ──
# Case 1: Known rewards
_rewards = torch.tensor([[1.0, 0.0, 1.0, 0.0]])  # batch=1, G=4
_adv = compute_group_advantages(_rewards)
assert _adv.shape == (1, 4), f'Expected shape (1,4), got {_adv.shape}'
assert torch.allclose(_adv.mean(dim=-1), torch.zeros(1), atol=1e-6), \
    'Advantages should have zero mean within each group'

# Case 2: All same rewards → advantages should be 0
_same = torch.tensor([[1.0, 1.0, 1.0, 1.0]])
_adv_same = compute_group_advantages(_same)
assert torch.allclose(_adv_same, torch.zeros_like(_adv_same), atol=1e-6), \
    'Equal rewards should yield zero advantages'

# Case 3: Batch of 2 groups
_batch = torch.tensor([[1.0, 0.0], [0.5, 0.5]])
_adv_batch = compute_group_advantages(_batch)
assert _adv_batch.shape == (2, 2)
assert torch.allclose(_adv_batch[1], torch.zeros(2), atol=1e-6), 'Second group is uniform'
assert _adv_batch[0, 0] > 0, 'Higher reward should have positive advantage'
print('✓ Exercise 2 passed!')


### Exercise 3: Importance Sampling Ratios

The policy ratio measures how much the current policy $\pi_\theta$ differs from
the old policy $\pi_{\theta_{\text{old}}}$ at each token position:

$$\rho_{i,t}(\theta) = \frac{\pi_\theta(o_{i,t} \mid q, o_{i,<t})}{\pi_{\theta_{\text{old}}}(o_{i,t} \mid q, o_{i,<t})} = \exp\big(\log \pi_\theta(o_{i,t}) - \log \pi_{\theta_{\text{old}}}(o_{i,t})\big)$$

**Input:**
- `new_log_probs`: Tensor of shape `(batch_size, seq_len)` — log-probs under $\pi_\theta$
- `old_log_probs`: Tensor of shape `(batch_size, seq_len)` — log-probs under $\pi_{\theta_{\text{old}}}$

**Output:**
- `ratios`: Tensor of shape `(batch_size, seq_len)`

In [ ]:
def compute_policy_ratios(
    new_log_probs: torch.Tensor,
    old_log_probs: torch.Tensor,
) -> torch.Tensor:
    """
    Compute importance sampling ratios.

    Args:
        new_log_probs: (batch_size, seq_len) — log probs under current policy
        old_log_probs: (batch_size, seq_len) — log probs under old policy
    Returns:
        ratios: (batch_size, seq_len)
    """
    ### SOLUTION ###
    return torch.exp(new_log_probs - old_log_probs)
    ### END SOLUTION ###


**Test your implementation:**

In [ ]:
# ── Test: compute_policy_ratios ──
# Case 1: Same policy → ratio should be 1
_lp = torch.tensor([[-1.0, -2.0, -0.5]])
_ratios = compute_policy_ratios(_lp, _lp)
assert torch.allclose(_ratios, torch.ones_like(_ratios), atol=1e-6), \
    'Same log-probs should give ratio=1'

# Case 2: Known values
_new_lp = torch.tensor([[-1.0, -2.0]])
_old_lp = torch.tensor([[-1.5, -1.5]])
_r = compute_policy_ratios(_new_lp, _old_lp)
_expected_r = torch.exp(_new_lp - _old_lp)
assert torch.allclose(_r, _expected_r, atol=1e-6), 'Ratios do not match expected values'
assert (_r > 0).all(), 'Ratios must be positive'
print('✓ Exercise 3 passed!')


### Exercise 4: Clipped Surrogate Objective

The clipped surrogate prevents large policy updates by restricting the effective ratio:

$$L_{i,t}^{\text{clip}}(\theta) = \min\big(\rho_{i,t}\, \hat{A}_i, \;\text{clip}(\rho_{i,t}, 1-\epsilon, 1+\epsilon)\, \hat{A}_i\big)$$

**Intuition:**
- When $\hat{A}_i > 0$ (good action), the ratio is clipped at $1+\epsilon$ — we don't let the
  policy move *too far* in the rewarding direction.
- When $\hat{A}_i < 0$ (bad action), the ratio is clipped at $1-\epsilon$ — we don't over-penalize.

**Input:**
- `ratios`: `(batch_size, seq_len)`
- `advantages`: `(batch_size,)` — one advantage per output (broadcast over tokens)
- `clip_eps`: clipping parameter (default 0.2)

**Output:**
- `objective`: `(batch_size, seq_len)` — per-token clipped objective

In [ ]:
def compute_clipped_objective(
    ratios: torch.Tensor,
    advantages: torch.Tensor,
    clip_eps: float = 0.2,
) -> torch.Tensor:
    """
    Compute the PPO-style clipped surrogate objective.

    Args:
        ratios: (batch_size, seq_len) — importance sampling ratios
        advantages: (batch_size,) — per-output advantages
        clip_eps: clipping threshold
    Returns:
        objective: (batch_size, seq_len)
    """
    ### SOLUTION ###
    adv = advantages.unsqueeze(-1)                             # (B, 1)
    unclipped = ratios * adv
    clipped = torch.clamp(ratios, 1 - clip_eps, 1 + clip_eps) * adv
    return torch.min(unclipped, clipped)
    ### END SOLUTION ###


**Test your implementation:**

In [ ]:
# ── Test: compute_clipped_objective ──
# Case 1: Ratio within clip range → min = unclipped
_ratios = torch.tensor([[1.0, 1.1]])
_adv = torch.tensor([1.0])
_obj = compute_clipped_objective(_ratios, _adv, clip_eps=0.2)
assert _obj.shape == (1, 2)
assert torch.allclose(_obj, _ratios * _adv.unsqueeze(-1), atol=1e-6), \
    'Within clip range, objective = ratio * advantage'

# Case 2: Ratio > 1+eps with positive advantage → clipped
_ratios2 = torch.tensor([[1.5]])  # exceeds 1+0.2=1.2
_adv2 = torch.tensor([1.0])
_obj2 = compute_clipped_objective(_ratios2, _adv2, clip_eps=0.2)
_expected2 = torch.tensor([[1.2]])  # clip(1.5, 0.8, 1.2) * 1.0 = 1.2
assert torch.allclose(_obj2, _expected2, atol=1e-6), \
    f'Expected {_expected2.item()}, got {_obj2.item()}'

# Case 3: Ratio < 1-eps with negative advantage → clipped
_ratios3 = torch.tensor([[0.5]])  # below 1-0.2=0.8
_adv3 = torch.tensor([-1.0])
_obj3 = compute_clipped_objective(_ratios3, _adv3, clip_eps=0.2)
_expected3 = torch.tensor([[-0.8]])  # min(-0.5, clip(0.5,0.8,1.2)*-1) = min(-0.5, -0.8) = -0.8
assert torch.allclose(_obj3, _expected3, atol=1e-6), \
    f'Expected {_expected3.item()}, got {_obj3.item()}'
print('✓ Exercise 4 passed!')


### Exercise 5: KL Divergence Penalty

GRPO penalizes deviation from a reference policy $\pi_{\text{ref}}$ using a per-token
KL divergence **estimator**:

$$D_{\text{KL}}^{(t)} = e^{\delta_t} - \delta_t - 1, \quad \delta_t = \log \pi_{\text{ref}}(o_t) - \log \pi_\theta(o_t)$$

**Properties of this estimator:**
- $D_{\text{KL}}^{(t)} \geq 0$ for all $\delta_t$ (by convexity of $e^x - x - 1$)
- $D_{\text{KL}}^{(t)} = 0$ when $\pi_\theta = \pi_{\text{ref}}$
- It is an unbiased estimator of $\text{KL}(\pi_\theta \| \pi_{\text{ref}})$

**Input:**
- `log_probs_policy`: `(batch_size, seq_len)` — log-probs under $\pi_\theta$
- `log_probs_ref`: `(batch_size, seq_len)` — log-probs under $\pi_{\text{ref}}$

**Output:**
- `kl`: `(batch_size, seq_len)` — per-token KL estimate

In [ ]:
def compute_kl_divergence(
    log_probs_policy: torch.Tensor,
    log_probs_ref: torch.Tensor,
) -> torch.Tensor:
    """
    Compute the per-token KL divergence estimate.

    Args:
        log_probs_policy: (batch_size, seq_len)
        log_probs_ref: (batch_size, seq_len)
    Returns:
        kl: (batch_size, seq_len) — non-negative KL estimate
    """
    ### SOLUTION ###
    delta = log_probs_ref - log_probs_policy
    return torch.exp(delta) - delta - 1
    ### END SOLUTION ###


**Test your implementation:**

In [ ]:
# ── Test: compute_kl_divergence ──
# Case 1: Same distribution → KL = 0
_lp = torch.tensor([[-1.0, -2.0, -0.5]])
_kl = compute_kl_divergence(_lp, _lp)
assert torch.allclose(_kl, torch.zeros_like(_kl), atol=1e-6), 'Same dist → KL=0'

# Case 2: KL must be non-negative
torch.manual_seed(0)
_policy = torch.randn(5, 10) - 2.0  # random log probs
_ref = torch.randn(5, 10) - 2.0
_kl2 = compute_kl_divergence(_policy, _ref)
assert (_kl2 >= -1e-6).all(), 'KL divergence must be non-negative'

# Case 3: Known value
_p = torch.tensor([[-1.0]])
_r = torch.tensor([[-2.0]])
_kl3 = compute_kl_divergence(_p, _r)
_delta = -2.0 - (-1.0)  # = -1.0
_expected_kl = torch.exp(torch.tensor(_delta)) - _delta - 1  # e^{-1} + 1 - 1 = 1/e ≈ 0.368
assert torch.allclose(_kl3, _expected_kl.unsqueeze(0).unsqueeze(0), atol=1e-5), \
    f'Expected {_expected_kl.item():.4f}, got {_kl3.item():.4f}'
print('✓ Exercise 5 passed!')


### Exercise 6: Full GRPO Loss Function

Combine all components into the complete GRPO loss. Given a batch of prompts with
$G$ sampled outputs each:

$$\mathcal{L}_{\text{GRPO}} = -\frac{1}{B \cdot G} \sum_{b=1}^{B} \sum_{i=1}^{G}
\frac{1}{|o_{b,i}|} \sum_{t=1}^{|o_{b,i}|}
\Big( L_{b,i,t}^{\text{clip}} - \beta \cdot D_{\text{KL}}^{(b,i,t)} \Big)$$

Note the **negation** — we minimize the loss (i.e., maximize the objective).

**Input:** (all pre-computed)
- `new_log_probs`: `(B*G, T)` — current policy log-probs
- `old_log_probs`: `(B*G, T)` — old policy log-probs
- `ref_log_probs`: `(B*G, T)` — reference policy log-probs
- `rewards`: `(B, G)` — outcome-level rewards
- `response_mask`: `(B*G, T)` — binary mask (1 for response tokens, 0 for padding/prompt)
- `clip_eps`, `beta`: hyperparameters

**Output:**
- `loss`: scalar

In [ ]:
def compute_grpo_loss(
    new_log_probs: torch.Tensor,   # (B*G, T)
    old_log_probs: torch.Tensor,   # (B*G, T)
    ref_log_probs: torch.Tensor,   # (B*G, T)
    rewards: torch.Tensor,          # (B, G)
    response_mask: torch.Tensor,    # (B*G, T)
    clip_eps: float = 0.2,
    beta: float = 0.04,
) -> torch.Tensor:
    """
    Compute the full GRPO loss.

    Returns:
        loss: scalar tensor (to be minimized)
    """
    B_times_G, T = new_log_probs.shape
    G = rewards.shape[1]
    B = B_times_G // G

    ### SOLUTION ###
    # Step 1: Group-normalized advantages → (B*G,)
    advantages = compute_group_advantages(rewards)  # (B, G)
    advantages = advantages.reshape(-1)              # (B*G,)

    # Step 2: Importance sampling ratios (B*G, T)
    ratios = compute_policy_ratios(new_log_probs, old_log_probs)

    # Step 3: Clipped surrogate objective (B*G, T)
    clipped_obj = compute_clipped_objective(ratios, advantages, clip_eps)

    # Step 4: KL divergence penalty (B*G, T)
    kl = compute_kl_divergence(new_log_probs, ref_log_probs)

    # Step 5: Per-token objective
    objective_per_token = clipped_obj - beta * kl

    # Step 6: Masked mean per sequence
    masked_obj = objective_per_token * response_mask
    seq_lengths = response_mask.sum(dim=-1).clamp(min=1)
    per_seq_obj = masked_obj.sum(dim=-1) / seq_lengths  # (B*G,)

    # Step 7: Average and negate (we minimise the loss)
    loss = -per_seq_obj.mean()
    return loss
    ### END SOLUTION ###


**Test your implementation:**

In [ ]:
# ── Test: compute_grpo_loss ──
torch.manual_seed(42)
B, G, T = 2, 4, 5

# Create synthetic data
_new_lp = torch.randn(B * G, T) - 2.0
_old_lp = _new_lp + torch.randn_like(_new_lp) * 0.05  # slightly different
_ref_lp = _new_lp + torch.randn_like(_new_lp) * 0.1
_rewards = torch.tensor([[1.0, 0.0, 1.0, 0.0], [0.0, 1.0, 1.0, 0.0]])
_mask = torch.ones(B * G, T)
_mask[:, -1] = 0  # last token is padding

_loss = compute_grpo_loss(_new_lp, _old_lp, _ref_lp, _rewards, _mask)

assert _loss.dim() == 0, f'Loss should be a scalar, got dim={_loss.dim()}'
assert torch.isfinite(_loss), 'Loss must be finite'
print(f'GRPO loss: {_loss.item():.4f}')

# Sanity: when new_log_probs == old_log_probs, ratios=1, so clipped objective
# should be close to the advantage times 1
_loss_same = compute_grpo_loss(_old_lp, _old_lp, _ref_lp, _rewards, _mask)
assert torch.isfinite(_loss_same), 'Loss must be finite'
print(f'GRPO loss (same policy): {_loss_same.item():.4f}')
print('✓ Exercise 6 passed!')


---
## Part 3: Verifiable Rewards for Mathematics

RLVR uses programmatic verification instead of a learned reward model. For math problems,
the reward function:
1. **Extracts** the final answer from the model's response
2. **Compares** it to the ground-truth answer
3. Returns a binary reward (1 if correct, 0 otherwise)

Answer extraction must handle various formats:
- `\boxed{42}` (LaTeX boxed)
- `The answer is 42`
- `**42**`
- Just a number at the end of the response

### Exercise 7: Answer Extraction and Reward Computation

Implement a function that extracts the numerical answer from a model's response.
AIME answers are always integers in $[0, 999]$.

**Strategy:**
1. Look for `\boxed{...}` patterns (highest priority)
2. Look for "the answer is ..." patterns
3. Fall back to the last number in the response

In [ ]:
def extract_answer(response: str) -> Optional[int]:
    """
    Extract the numerical answer from a model response.

    Args:
        response: the model's full text response
    Returns:
        Extracted integer answer, or None if no answer found.
    """
    ### SOLUTION ###
    # Strategy 1: Look for \\boxed{...}
    matches = re.findall(r'\\boxed\{(\d+)\}', response)
    if matches:
        return int(matches[-1])

    # Strategy 2: Look for 'the answer is <number>'
    match = re.search(r'[Tt]he\s+(?:final\s+)?answer\s+is\s*[:\s]*(\d+)', response)
    if match:
        return int(match.group(1))

    # Strategy 3: Fall back to the last integer in the response
    matches = re.findall(r'(\d+)', response)
    if matches:
        return int(matches[-1])

    return None
    ### END SOLUTION ###


def compute_reward(response: str, ground_truth: int) -> float:
    """Binary reward: 1.0 if extracted answer matches ground truth, else 0.0."""
    predicted = extract_answer(response)
    if predicted is None:
        return 0.0
    return 1.0 if predicted == ground_truth else 0.0


**Test your implementation:**

In [ ]:
# ── Test: extract_answer ──
assert extract_answer(r'Therefore $\boxed{42}$.') == 42, 'Should extract from \\boxed{}'
assert extract_answer('The answer is 123') == 123, 'Should extract from "the answer is"'
assert extract_answer('The final answer is: 7') == 7, 'Should handle "final answer is:"'
assert extract_answer('After computing, we get 99.') == 99, 'Should get last number'
assert extract_answer(r'First $\boxed{10}$, but actually $\boxed{20}$') == 20, \
    'Should return LAST boxed answer'
assert extract_answer('No numbers here') is None, 'Should return None if no number'

# Test compute_reward
assert compute_reward(r'$\boxed{42}$', 42) == 1.0
assert compute_reward(r'$\boxed{41}$', 42) == 0.0
assert compute_reward('I have no idea', 42) == 0.0
print('✓ Exercise 7 passed!')


---
### Exercise 8: Mini GRPO Training Step (Putting It All Together)

Now let's combine everything into a single GRPO training step. We'll use a small
toy model to demonstrate the full pipeline.

**Goal:** Given a batch of prompts, sample groups of responses, compute rewards,
and perform one gradient update.

We provide the scaffolding — you fill in the key steps.

In [ ]:
def grpo_training_step(
    model: torch.nn.Module,
    ref_model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    prompts: List[str],
    ground_truths: List[int],
    tokenizer,
    group_size: int = 4,
    clip_eps: float = 0.2,
    beta: float = 0.04,
    max_new_tokens: int = 512,
) -> Dict[str, float]:
    """
    Execute one GRPO training step.

    Returns dict with metrics: loss, mean_reward, mean_kl
    """
    B = len(prompts)
    G = group_size

    # ── Step 1: Sample G responses per prompt using the current policy ──
    all_responses = []
    for prompt in prompts:
        for _ in range(G):
            inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
            with torch.no_grad():
                output = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.9,
                )
            response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:])
            all_responses.append(response)

    ### SOLUTION ###
    # ── Step 2: Compute rewards (B, G) ──
    rewards_list = []
    for i in range(B):
        group_rewards = [
            compute_reward(all_responses[i * G + j], ground_truths[i])
            for j in range(G)
        ]
        rewards_list.append(group_rewards)
    rewards = torch.tensor(rewards_list, dtype=torch.float32).to(model.device)

    # ── Step 3: Tokenize prompt+response pairs and build response mask ──
    all_prompt_lens = []
    all_input_ids = []
    for i, prompt in enumerate(prompts):
        prompt_len = tokenizer(prompt, return_tensors='pt')['input_ids'].shape[1]
        for j in range(G):
            full_text = prompt + all_responses[i * G + j]
            enc = tokenizer(full_text, return_tensors='pt',
                            truncation=True, max_length=1024)
            all_input_ids.append(enc['input_ids'][0])
            all_prompt_lens.append(prompt_len)

    max_len = max(ids.shape[0] for ids in all_input_ids)
    padded_ids = torch.zeros(B * G, max_len, dtype=torch.long).to(model.device)
    attn_mask  = torch.zeros(B * G, max_len, dtype=torch.long).to(model.device)
    for k, ids in enumerate(all_input_ids):
        padded_ids[k, :len(ids)] = ids.to(model.device)
        attn_mask[k,  :len(ids)] = 1

    # Response mask aligned with shifted labels (T = max_len - 1)
    T = max_len - 1
    response_mask = torch.zeros(B * G, T, dtype=torch.float32).to(model.device)
    for k, plen in enumerate(all_prompt_lens):
        seq_len = int(attn_mask[k].sum().item()) - 1  # shift by 1 for labels
        response_mask[k, plen - 1 : seq_len] = 1.0   # -1 accounts for the shift

    labels = padded_ids[:, 1:]  # (B*G, T)

    # Old log-probs (detached) and ref log-probs
    with torch.no_grad():
        old_logits = model(padded_ids, attention_mask=attn_mask).logits[:, :-1]
        old_log_probs = compute_log_probs(old_logits, labels)

        ref_logits = ref_model(padded_ids, attention_mask=attn_mask).logits[:, :-1]
        ref_log_probs = compute_log_probs(ref_logits, labels)

    # New log-probs (with gradients)
    new_logits = model(padded_ids, attention_mask=attn_mask).logits[:, :-1]
    new_log_probs = compute_log_probs(new_logits, labels)

    # ── Step 4: GRPO loss ──
    loss = compute_grpo_loss(
        new_log_probs, old_log_probs, ref_log_probs,
        rewards, response_mask,
        clip_eps=clip_eps, beta=beta,
    )

    # ── Step 5: Backprop ──
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    return {'loss': loss.item(), 'mean_reward': rewards.mean().item()}
    ### END SOLUTION ###


> **Note:** Running a full GRPO training loop requires significant GPU memory.
> The function above is for understanding the algorithm structure.
> In practice, libraries like [TRL](https://github.com/huggingface/trl) or
> [OpenRLHF](https://github.com/OpenRLHF/OpenRLHF) handle the engineering
> complexity (distributed sampling, memory management, etc.).

---
## Part 4: Test-Time Compute Scaling

### 4.1 What is Test-Time Compute Scaling?

Instead of making models larger (train-time scaling), we can improve performance
by using **more computation at inference time**:

| Strategy | Description | Example |
|----------|-------------|----------|
| **Internal Thinking** | Model reasons before answering | `<think>...reasoning...</think>` |
| **Best-of-N** | Generate N answers, pick the best | Majority voting |
| **Search** | Explore reasoning paths with tree search | MCTS, beam search |

**Thinking models** (DeepSeek-R1, Qwen3, QwQ) implement the first strategy via
special thinking tokens. Qwen3 models are **hybrid** — they support both
thinking and non-thinking modes:

- `enable_thinking=True`: generates `<think>...internal reasoning...</think>` before the answer
- `enable_thinking=False`: directly generates the answer (faster, but less accurate on hard problems)

### 4.2 The Thinking Budget

The **thinking budget** controls how many tokens the model can spend on internal reasoning.
This creates a natural **accuracy-vs-latency tradeoff**:

- Small budget → fast but less accurate (shallow reasoning)
- Large budget → slow but more accurate (deep reasoning)
- The relationship is often **log-linear**: performance scales with log(thinking tokens)


### 4.3 Loading Qwen3-4B

We load the model once and switch between thinking and non-thinking modes via the
chat template.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen3-4B'

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.eval()
print('Model loaded!')


### 4.4 Generation Helpers

We provide a helper for **non-thinking** generation. You will implement the
**thinking with budget control** version.

In [ ]:
def generate_non_thinking(
    model, tokenizer, problem: str,
    max_new_tokens: int = 2048,
    temperature: float = 0.6,
) -> str:
    """Generate a response WITHOUT thinking (direct answer)."""
    messages = [{'role': 'user', 'content': problem}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, top_p=0.95, top_k=20,
            do_sample=True,
        )
    response_ids = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(response_ids, skip_special_tokens=True)


### Exercise 9: Generation with Thinking Budget Control

Implement generation with a **controllable thinking budget**. The approach:

1. Apply the chat template with `enable_thinking=True`
2. Generate tokens up to the thinking budget
3. If the model hasn't finished thinking (no `</think>` token yet), force-end the
   thinking phase by appending an early-stop suffix and continue generation
4. Parse out the thinking content and the final response

The `</think>` token in Qwen3 has token ID **151668**.

**Hint:** Use a two-phase generation strategy:
- Phase 1: Generate up to `thinking_budget` tokens
- Phase 2: If thinking not finished, append early-stop text and generate the answer

In [ ]:
THINK_END_TOKEN_ID = 151668  # </think> token ID in Qwen3

EARLY_STOP_SUFFIX = (
    '\nConsidering the limited time, I will give my answer based on '
    'the reasoning so far.\n</think>\n\n'
)


def generate_with_thinking_budget(
    model, tokenizer, problem: str,
    thinking_budget: int = 1024,
    max_new_tokens: int = 4096,
    temperature: float = 0.6,
) -> Tuple[str, str, int]:
    """
    Generate with a controlled thinking budget.

    Args:
        model: the Qwen3 model
        tokenizer: the tokenizer
        problem: the math problem text
        thinking_budget: max tokens for thinking phase
        max_new_tokens: max total new tokens (thinking + answer)
        temperature: sampling temperature

    Returns:
        (thinking_text, answer_text, num_thinking_tokens)
    """
    ### SOLUTION ###
    # Step 1: Prepare input with enable_thinking=True
    messages = [{'role': 'user', 'content': problem}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=True,
    )
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    prompt_len = inputs['input_ids'].shape[1]

    # Step 2: Phase 1 — generate up to thinking_budget tokens
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=thinking_budget,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
        )

    new_ids = output[0][prompt_len:].tolist()

    # Step 3 & 4: If </think> not found, force-end thinking
    if THINK_END_TOKEN_ID not in new_ids:
        suffix_ids = tokenizer(
            EARLY_STOP_SUFFIX, add_special_tokens=False
        )['input_ids']
        extended = torch.cat(
            [output, torch.tensor([suffix_ids], device=model.device)], dim=1
        )
        with torch.no_grad():
            output = model.generate(
                extended,
                max_new_tokens=max_new_tokens - thinking_budget,
                do_sample=True,
                temperature=temperature,
                top_p=0.95,
            )

    # Step 5: Parse output
    all_new_ids = output[0][prompt_len:].tolist()
    if THINK_END_TOKEN_ID in all_new_ids:
        split_pos = all_new_ids.index(THINK_END_TOKEN_ID)
        thinking_ids = all_new_ids[:split_pos]
        answer_ids   = all_new_ids[split_pos + 1:]
    else:
        thinking_ids = all_new_ids
        answer_ids   = []

    thinking_text     = tokenizer.decode(thinking_ids, skip_special_tokens=True)
    answer_text       = tokenizer.decode(answer_ids,   skip_special_tokens=True)
    num_thinking_tokens = len(thinking_ids)

    return thinking_text, answer_text, num_thinking_tokens
    ### END SOLUTION ###


**Quick test** (requires model to be loaded):

In [ ]:
# ── Test: generate_with_thinking_budget ──
_problem = 'Find the sum of all integer bases $b>9$ for which $17_b$ is a divisor of $97_b.$'
_budget = 1024
_think, _answer, _n_tokens = generate_with_thinking_budget(
    model, tokenizer, _problem, thinking_budget=_budget
)
print(f'Thinking ({_n_tokens} tokens): {_think[:200]}...')
print(f'Answer: {_answer}')
assert _n_tokens <= _budget + 50, 'Thinking should be within budget (with small tolerance)'
print('✓ Exercise 9 basic test passed!')

---
## Part 5: Evaluation on AIME 2025

### 5.1 Loading the Benchmarks

We load AIME problems from HuggingFace datasets. Each problem has:
- A problem statement (text)
- A ground-truth integer answer (0–999)

AIME 2025 contains 30 problems (I & II).

In [ ]:
from datasets import load_dataset

try:
    aime25_ds = load_dataset('opencompass/AIME2025', 'AIME2025-I', split='test')
    # Adapt column names (may vary by dataset)
    cols = aime25_ds.column_names
    problem_col = 'problem' if 'problem' in cols else 'Problem' if 'Problem' in cols else cols[0]
    answer_col = 'answer' if 'answer' in cols else 'Answer' if 'Answer' in cols else cols[-1]
    aime25 = [{'id': f'2025-{i}', 'problem': row[problem_col], 'answer': int(row[answer_col])}
              for i, row in enumerate(aime25_ds)]
    print(f'Loaded AIME 2025: {len(aime25)} problems')
except Exception as e:
    print(f'Could not load AIME 2025 from HuggingFace: {e}')
    print('Using built-in subset...')
    aime25 = [
        {'id': '2025-I-1', 'answer': 468,
         'problem': 'Find the sum of all integer bases b > 9 for which 17_b is a '
         'divisor of 97_b.'},
        {'id': '2025-I-2', 'answer': 58,
         'problem': 'Let S be the set of all positive integers that have four digits '
         'and have all nonzero digits. Let f(n) denote the sum of the digits of n. '
         'Find the remainder when the sum of f(n) over all n in S is divided by 1000.'},
        {'id': '2025-I-3', 'answer': 225,
         'problem': 'The 9 cells of a 3x3 grid are each colored one of two colors. '
         'Find the number of colorings such that no 2x2 square is a single color.'},
    ]

print(f'\nTotal: {len(aime25)} AIME25 problems')
print(f'Example: {aime25[0]["id"]} → answer = {aime25[0]["answer"]}')


### Exercise 10: Evaluation Pipeline

Implement the evaluation loop that runs the model on AIME problems with different
thinking budgets and records accuracy.

**Your task:** Fill in the evaluation function that:
1. Iterates over problems
2. Generates responses (thinking or non-thinking)
3. Extracts answers and computes accuracy

In [ ]:
from tqdm import tqdm


def evaluate_on_aime(
    model, tokenizer,
    problems: List[Dict],
    mode: str = 'thinking',       # 'thinking' or 'non_thinking'
    thinking_budget: int = 1024,   # only used if mode='thinking'
    max_new_tokens: int = 4096,
    temperature: float = 0.6,
) -> Dict:
    """
    Evaluate a model on a set of AIME problems.

    Args:
        model, tokenizer: the model and tokenizer
        problems: list of dicts with 'problem' and 'answer' keys
        mode: 'thinking' (with budget) or 'non_thinking'
        thinking_budget: max thinking tokens (if mode='thinking')

    Returns:
        dict with 'accuracy', 'correct', 'total', 'results' (per-problem details)
    """
    results = []

    ### SOLUTION ###
    for prob in tqdm(problems, desc=f'Evaluating ({mode})'):
        prompt = (
            f'Solve the following math problem. '
            f'Provide your final answer as a single integer inside \\boxed{{}}.'
            f'\n\n{prob["problem"]}'
        )

        thinking_tokens = 0
        if mode == 'non_thinking':
            response = generate_non_thinking(
                model, tokenizer, prompt,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
            )
            answer_text = response
        else:  # 'thinking'
            _, answer_text, thinking_tokens = generate_with_thinking_budget(
                model, tokenizer, prompt,
                thinking_budget=thinking_budget,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
            )

        predicted = extract_answer(answer_text)
        correct   = predicted == prob['answer']

        results.append({
            'id':             prob.get('id', ''),
            'predicted':      predicted,
            'ground_truth':   prob['answer'],
            'correct':        correct,
            'thinking_tokens': thinking_tokens,
        })
    ### END SOLUTION ###

    correct = sum(1 for r in results if r['correct'])
    total = len(results)
    accuracy = correct / total if total > 0 else 0.0

    return {
        'accuracy': accuracy,
        'correct': correct,
        'total': total,
        'results': results,
    }


### 5.2 Running the Experiments

We evaluate Qwen3-4B under multiple configurations:
1. **Non-thinking** mode (baseline)
2. **Thinking** mode with budgets: 128, 256, 512, 1024, 2048, 4096 tokens

This may take a while depending on your hardware. Adjust `BUDGETS` or
use a subset of problems for faster iteration.

In [ ]:
# ── Configuration ──
BUDGETS = [128, 256, 512, 1024, 2048, 4096]

# Use a subset for faster experimentation (set to None for all problems)
MAX_PROBLEMS = 5  # e.g., 10 for quick testing
aime25_subset = aime25[:MAX_PROBLEMS] if MAX_PROBLEMS else aime25

results = {'aime25': {}}

# ── Non-thinking baseline ──
print('=== Non-Thinking Mode ===')
for dataset_name, problems in [('aime25', aime25_subset)]:
    res = evaluate_on_aime(model, tokenizer, problems, mode='non_thinking')
    results[dataset_name]['non_thinking'] = res
    print(f'{dataset_name} non-thinking: {res["correct"]}/{res["total"]} '
          f'= {res["accuracy"]:.1%}')

# ── Thinking with different budgets ──
for budget in BUDGETS:
    print(f'\n=== Thinking Budget: {budget} tokens ===')
    for dataset_name, problems in [('aime25', aime25_subset)]:
        res = evaluate_on_aime(
            model, tokenizer, problems,
            mode='thinking', thinking_budget=budget,
            max_new_tokens=(budget+1024)
        )
        results[dataset_name][f'thinking_{budget}'] = res
        print(f'{dataset_name} budget={budget}: {res["correct"]}/{res["total"]} '
              f'= {res["accuracy"]:.1%}')

print('\nDone!')


### 5.3 Visualizing the Results

Let's create two visualizations:
1. **Accuracy vs Thinking Budget** for AIME 2025
2. **Comparison bar chart** at selected budgets

In [ ]:
def plot_thinking_budget_scaling(results: dict, budgets: list):
    """Plot accuracy vs thinking budget for AIME 2025."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    for ax, (dataset_name, dataset_label) in zip(
        axes, [('aime25', 'AIME 2025')]
    ):
        ds_results = results[dataset_name]

        # Non-thinking baseline
        baseline_acc = ds_results['non_thinking']['accuracy'] * 100

        # Thinking accuracies
        thinking_accs = []
        for b in budgets:
            key = f'thinking_{b}'
            if key in ds_results:
                thinking_accs.append(ds_results[key]['accuracy'] * 100)
            else:
                thinking_accs.append(None)

        # Plot baseline as horizontal line
        ax.axhline(y=baseline_acc, color='gray', linestyle='--', linewidth=1.5,
                   label=f'Non-Thinking ({baseline_acc:.1f}%)')

        # Plot thinking budget curve
        valid = [(b, a) for b, a in zip(budgets, thinking_accs) if a is not None]
        if valid:
            bs, accs = zip(*valid)
            ax.plot(bs, accs, 'o-', color='#2196F3', linewidth=2, markersize=8,
                    label='Thinking Mode')

            # Annotate each point
            for b, a in zip(bs, accs):
                ax.annotate(f'{a:.1f}%', (b, a), textcoords='offset points',
                           xytext=(0, 10), ha='center', fontsize=9)

        ax.set_xscale('log', base=2)
        ax.set_xlabel('Thinking Budget (tokens)', fontsize=12)
        ax.set_ylabel('Accuracy (%)', fontsize=12)
        ax.set_title(f'{dataset_label}', fontsize=14, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 100)
        ax.set_xticks(budgets)
        ax.set_xticklabels([str(b) for b in budgets])

    fig.suptitle('Test-Time Compute Scaling: Qwen3-4B on AIME25',
                fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('thinking_budget_scaling.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: thinking_budget_scaling.png')


plot_thinking_budget_scaling(results, BUDGETS)


In [ ]:
def plot_comparison_bar(results: dict, budgets_to_show: list = [512, 2048]):
    """Bar chart comparing non-thinking vs thinking at selected budgets."""
    datasets = ['aime25']
    labels = ['AIME 2025']
    configs = ['non_thinking'] + [f'thinking_{b}' for b in budgets_to_show]
    config_labels = ['Non-Thinking'] + [f'Thinking\n(budget={b})' for b in budgets_to_show]
    colors = ['#9E9E9E', '#42A5F5', '#1565C0'][:len(configs)]

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(datasets))
    width = 0.8 / len(configs)

    for i, (cfg, cfg_label, color) in enumerate(zip(configs, config_labels, colors)):
        accs = []
        for ds in datasets:
            if cfg in results[ds]:
                accs.append(results[ds][cfg]['accuracy'] * 100)
            else:
                accs.append(0)
        bars = ax.bar(x + i * width - 0.4 + width/2, accs, width,
                      label=cfg_label, color=color, edgecolor='white')
        for bar, acc in zip(bars, accs):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{acc:.1f}%', ha='center', va='bottom', fontsize=9)

    ax.set_ylabel('Accuracy (%)', fontsize=12)
    ax.set_title('Thinking vs Non-Thinking: Qwen3-4B', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=12)
    ax.legend(fontsize=10)
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('thinking_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: thinking_comparison.png')


plot_comparison_bar(results)


---
## Part 6: Analysis Questions

Answer the following questions based on your experiments. Write your answers in
the markdown cells below each question.

### Question 1: GRPO vs PPO

What are the **two main advantages** of GRPO over PPO in the context of training
language models? What is the key assumption that makes the group-based advantage
estimation work?

**Answer:**

**1. Two main advantages of GRPO over PPO:**

1. **No critic network required.** PPO needs a separate value function (critic) trained alongside the policy to estimate baselines. GRPO replaces this with group-relative normalization: it samples $G$ outputs for the same prompt and uses the group mean reward as the baseline. This halves the memory footprint and eliminates the instability caused by an imperfect value estimator.

2. **More stable, interpretable advantage estimates.** Because all $G$ samples share the same prompt, the normalized advantages directly reflect *relative* quality within that group, reducing variance from prompt-to-prompt difficulty differences. PPO's value network must generalize across all prompts, which is harder to train.

**Key assumption:** The $G$ outputs sampled for a given prompt are approximately i.i.d. under the current policy, so their empirical mean and standard deviation are reasonable estimates of the expected reward and its variability for that prompt. This breaks down if $G$ is too small or if the reward distribution is highly multi-modal.


### Question 2: Reward Sparsity

In RLVR for math, the reward is binary (0 or 1). What problems does this create
for training? How does the group size $G$ help mitigate this? What happens if
all $G$ samples for a prompt receive the same reward?

**Answer:**

**Problems created by binary rewards:**
- **Gradient sparsity:** Early in training the model rarely solves any problem, so most sequences get reward $0$. With no positive signal the gradient is essentially zero and learning stalls.
- **No partial credit:** A response that is almost correct gets the same reward (0) as a completely wrong one, giving the optimizer no gradient direction toward near-correct solutions.
- **High variance:** Whether a group of $G$ responses happens to include one correct answer is very noisy for small $G$.

**How group size $G$ helps:**
- Larger $G$ increases the probability that at least one response in the group is correct, guaranteeing nonzero reward variance within the group. This means normalized advantages are nonzero more often, providing training signal even when the task is hard.
- With $G=1$ the advantage is trivially 0 (nothing to normalize). With $G \ge 4$ there is generally enough diversity.

**When all $G$ samples have the same reward:**
The group standard deviation is 0, so the normalized advantage $\frac{r - \bar{r}}{\sigma + \varepsilon}$ collapses to 0 for every token in the group. The GRPO gradient for that batch element is zero — the model learns nothing from prompts where it either always succeeds or always fails.


### Question 3: KL Divergence Role

Why does GRPO include a KL penalty against a reference policy $\pi_{\text{ref}}$?
What would happen if we set $\beta = 0$? What would happen if $\beta$ is too large?

**Answer:**

**Role of the KL penalty:**
The KL term $\beta \cdot D_{KL}(\pi_\theta \| \pi_{\text{ref}})$ regularizes the policy to stay close to the reference model (typically the SFT checkpoint). This serves two purposes:
1. **Stability:** Prevents large destructive updates that could cause reward hacking or catastrophic forgetting.
2. **Language quality:** Keeps the model's outputs fluent and coherent, since the reference model has already learned to produce high-quality natural language.

**If $\beta = 0$:**
No regularization. The policy is free to drift arbitrarily far from the reference. In practice this leads to *reward hacking*—the model finds degenerate strategies (e.g., repeating the same token, outputting only numbers) that achieve high reward on the verifiable metric while losing general language ability. Training can also become unstable.

**If $\beta$ is too large:**
The KL penalty dominates the loss. Every gradient step is mostly spent keeping the policy close to the reference, leaving little room to improve on the actual task reward. Learning becomes extremely slow or effectively stops. The model behaves nearly identically to the reference regardless of the reward signal.


### Question 4: Thinking Budget Analysis

Based on your experiments:
- At what thinking budget does the model start outperforming non-thinking mode?
- Is the improvement from thinking budget linear, logarithmic, or something else?
- Is there a point of **diminishing returns**? Why might this happen?

**Answer** *(based on typical Qwen3-4B results; your numbers may vary slightly):*

**At what budget does thinking outperform non-thinking?**
Typically around **512–1024 tokens**. Below this threshold the model does not have enough budget to complete a meaningful chain-of-thought, and the forced early termination produces noisy outputs that hurt accuracy.

**Shape of the improvement curve:**
The improvement is approximately **logarithmic** — accuracy rises quickly from 128→512 tokens, then gains slow considerably beyond 1024–2048 tokens.

**Diminishing returns:**
Yes, there is a clear plateau around 2048–4096 tokens. Two reasons:
1. Most correct solutions on AIME can be expressed within ~1500 reasoning tokens; extra tokens are spent re-checking or exploring dead ends.
2. The model's reasoning quality is bounded by its parameters; more tokens allow refinement but not fundamentally new capabilities.


### Question 5: Practical Considerations

You are deploying a math-solving model as a product. You need to balance accuracy
and latency. Based on your results:
- What thinking budget would you recommend for a **real-time tutoring** application?
- What about for an **offline competition solver**?
- How would you implement an **adaptive** thinking budget that uses more tokens
  for harder problems?

**Answer:**

**Real-time tutoring application:**
A budget of ~**512 tokens** (≈0.5–1 s on an A100). This provides a good step-by-step explanation without noticeable lag. Accuracy is materially better than non-thinking, and the partial reasoning trace is itself pedagogically useful for the student.

**Offline competition solver:**
Use the maximum affordable budget, e.g. **4096 tokens**, combined with majority voting over several samples (Best-of-8 or Best-of-16). Latency is irrelevant; maximising accuracy is the sole objective.

**Adaptive thinking budget:**

Several strategies exist:

1. **Confidence-based extension:** Generate with a small budget (e.g., 256 tokens). If `extract_answer` returns a high-confidence match (e.g., the same integer appears in ≥3 of the last 5 sentences), stop. Otherwise double the budget and continue.

2. **Difficulty prediction:** Train a small classifier on problem text embeddings to predict difficulty (easy / medium / hard) and map each tier to a fixed budget (256 / 1024 / 4096).

3. **Budget tokens prompt:** Instruct the model with explicit budget guidance in the system prompt (e.g., "You have at most 512 tokens to think"). Qwen3 is trained to respect such guidance, adjusting the depth of its reasoning accordingly.

4. **Iterative refinement:** Start with non-thinking mode. If the extracted answer is `None` or the model expresses uncertainty, re-run in thinking mode with a progressively larger budget until an answer is found or the maximum budget is reached.


---
## Bonus: Majority Voting Evaluation

As an extension, implement **majority voting** over $N$ samples. For each problem:
1. Generate $N$ responses
2. Extract the answer from each
3. Return the most common answer (breaking ties randomly)

Compare the accuracy of majority voting (non-thinking) vs single-pass thinking.
How many non-thinking samples are needed to match a single thinking pass?

In [ ]:
def evaluate_majority_voting(
    model, tokenizer,
    problems: List[Dict],
    n_samples: int = 8,
    mode: str = 'non_thinking',
    thinking_budget: int = 1024,
    temperature: float = 0.7,
) -> Dict:
    """
    Evaluate using majority voting over N samples.

    Returns:
        dict with 'accuracy', 'results'
    """
    from collections import Counter
    results = []

    for prob in tqdm(problems, desc=f'Majority@{n_samples}'):
        prompt = (f'Solve the following math problem. Provide your final answer '
                  f'as a single integer inside \\boxed{{}}.\n\n{prob["problem"]}')

        answers = []
        for _ in range(n_samples):
            if mode == 'non_thinking':
                response = generate_non_thinking(
                    model, tokenizer, prompt, temperature=temperature
                )
            else:
                _, response, _ = generate_with_thinking_budget(
                    model, tokenizer, prompt,
                    thinking_budget=thinking_budget, temperature=temperature
                )
            ans = extract_answer(response)
            if ans is not None:
                answers.append(ans)

        # Majority vote
        if answers:
            majority = Counter(answers).most_common(1)[0][0]
        else:
            majority = None

        correct = majority == prob['answer']
        results.append({
            'id': prob['id'],
            'predicted': majority,
            'ground_truth': prob['answer'],
            'correct': correct,
            'all_answers': answers,
        })

    correct_count = sum(1 for r in results if r['correct'])
    total = len(results)
    return {'accuracy': correct_count / total, 'correct': correct_count,
            'total': total, 'results': results}

majority_results = evaluate_majority_voting(
    model, tokenizer, aime25[:5], n_samples=8, mode='non_thinking'
)
print(f'Majority@8 accuracy: {majority_results["accuracy"]:.1%}')


---
**Congratulations!** You have:
- Implemented the core components of GRPO from scratch
- Built a verifiable reward function for mathematical reasoning
- Explored test-time compute scaling with thinking budget control
- Evaluated and visualized thinking vs non-thinking performance on AIME

These techniques are at the core of recent breakthroughs like DeepSeek-R1,
Qwen3, and other frontier reasoning models.